<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_11_1_debugging_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
**Module 11: Troubleshooting and Evaluating PyTorch Models**  

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 11 Material

* **Part 11.1: Debugging with PyTorch Networks** [[Notebook]](t81_558_class_11_1_debugging_pytorch.ipynb)
* Part 11.2: Critical PyTorch Errors [[Notebook]](t81_558_class_11_2_critical_errors.ipynb)
* Part 11.3: Overfitting and Underfitting [[Notebook]](t81_558_class_11_3_overfitting.ipynb)
* Part 11.4: Vanishing and Exploding Gradients [[Notebook]](t81_558_class_11_4_gradients.ipynb)
* Part 11.5: Error Metrics Beyond Accuracy [[Notebook]](t81_558_class_11_5_metrics.ipynb)

# Google CoLab Instructions

The following code checks that Google CoLab is running and sets up the correct hardware settings for PyTorch.

In [1]:
try:
    import google.colab
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Make use of a GPU or MPS (Apple) if one is available.  (see module 2.5)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Note: not using Google CoLab


Using device: mps


# Part 11.1: Debugging with PyTorch Networks

Debugging a neural network is different from debugging ordinary software. In most programs, a bug produces an obvious failure: an exception, a crash, or plainly wrong output. Neural networks fail this way too, but they also fail in a quieter and more dangerous way. The code runs without error, the loss prints, the training loop completes, and yet the network learns nothing useful. A misplaced activation, a target that is off by one dimension, or a layer that is silently disconnected from the loss will not raise an exception. It will simply produce a model that does not work.

Because of this, effective debugging in PyTorch is less about reading tracebacks and more about *inspecting* the network while it runs. The goal is to make the invisible visible: to look at the shapes flowing between layers, the values of the activations, the gradients after a backward pass, and the loss on a tiny slice of data you fully understand. This section introduces a small, reliable toolkit for that inspection. None of these techniques require special libraries; they use only PyTorch and a few well-placed print statements.

The techniques build on one another. We start with the single most common source of bugs, tensor shapes, then move to inspecting the model itself, observing intermediate activations with hooks, verifying that gradients actually flow, and finally the single most valuable sanity check in deep learning: overfitting one batch on purpose.

## Start With Shapes

The majority of PyTorch bugs are shape bugs. A tensor that should be `(batch, features)` arrives as `(features, batch)`, a channel dimension goes missing after a reshape, or a target vector is `(batch, 1)` when the loss function expects `(batch,)`. Many of these do not raise an error, because broadcasting quietly makes the operation "work" while computing something meaningless.

The defense is simple and it should become a habit: print the shape of a tensor whenever you are unsure of it, and print the shape of your model's output before you train anything. A single forward pass on a small batch, followed by a look at the input and output shapes, catches an enormous fraction of bugs before they waste a training run.

The example below defines a small classifier and does three things every debugging session should start with: it prints the model, counts its parameters, and confirms the output shape for a known input.

In [2]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# A small synthetic classification problem: 8 features, 3 classes.
X = torch.randn(256, 8)
y = torch.randint(0, 3, (256,))


class Net(nn.Module):
    def __init__(self, in_features=8, hidden=32, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, num_classes)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.out(x)


model = Net()

# 1. Print the model to confirm its structure matches your intent.
print(model)

# 2. Count parameters -- a quick check that the model is the size you expect.
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total}")
print(f"Trainable parameters: {trainable}")

# 3. Confirm the output shape for a small batch BEFORE training anything.
logits = model(X[:4])
print("\nInput batch shape: ", tuple(X[:4].shape))
print("Output logit shape:", tuple(logits.shape))

Net(
  (fc1): Linear(in_features=8, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=32, bias=True)
  (out): Linear(in_features=32, out_features=3, bias=True)
  (act): ReLU()
)

Total parameters:     1443
Trainable parameters: 1443

Input batch shape:  (4, 8)
Output logit shape: (4, 3)


The output confirms three things at a glance. The printed module shows the layers are wired in the order we intended. The parameter count is a sanity check on model size; a number that is far larger or smaller than expected usually means a dimension is wrong. And the output shape `(4, 3)` confirms that four inputs produce four rows of three class logits, exactly what a three-class classifier should emit. Establishing these facts before training means that if learning later goes wrong, we can rule out the model's basic structure as the cause.

## Inspecting Activations With Forward Hooks

Printing the final output shape tells you whether the network as a whole is wired correctly, but it does not tell you what is happening *inside*. When a shape error appears deep in a network, or when activations quietly collapse to all zeros or explode to very large values, you need to observe the intermediate tensors. PyTorch provides **forward hooks** for exactly this.

A forward hook is a function you attach to any submodule. PyTorch calls it every time that submodule runs, passing in the module, its inputs, and its output. The hook can record or print whatever you need without any change to the model's `forward` method. This is far cleaner than editing the model to add print statements, and it works even for models you did not write, such as pretrained networks from `torchvision`.

The example registers a hook on every linear layer to record the shape of its output, runs one forward pass, and then removes the hooks. Removing them afterward matters: a hook left in place keeps firing on every future forward pass, slowing training and cluttering output.

In [3]:
# A forward hook observes the input and output of a submodule without
# modifying the forward() method. Here we record each Linear layer's
# output shape as data flows through the network.
activation_shapes = {}


def make_hook(name):
    def hook(module, inputs, output):
        activation_shapes[name] = tuple(output.shape)
    return hook


handles = []
for name, layer in model.named_modules():
    if isinstance(layer, nn.Linear):
        handles.append(layer.register_forward_hook(make_hook(name)))

# Run one forward pass; the hooks fire and record shapes as a side effect.
_ = model(X[:16])

for name, shape in activation_shapes.items():
    print(f"{name:5s} output shape: {shape}")

# Always remove hooks when done so they do not fire on later passes.
for h in handles:
    h.remove()
print("\nhooks removed:", len(handles))

fc1   output shape: (16, 32)
fc2   output shape: (16, 32)
out   output shape: (16, 3)

hooks removed: 3


Reading the shapes from top to bottom traces the data's path through the network. The batch of sixteen samples enters `fc1` and leaves as `(16, 32)`, passes through `fc2` unchanged in width, and finally `out` reduces it to `(16, 3)`. If any of these were wrong, we would see exactly which layer introduced the problem, rather than only seeing a confusing error at the very end. The same hook pattern generalizes: instead of recording shapes, a hook can record the mean and standard deviation of activations, which is how you detect layers whose outputs have collapsed or saturated.

## Checking That Gradients Flow

A network learns only through the gradients computed during the backward pass. If a parameter's gradient is `None`, that parameter is disconnected from the loss and will never update, no matter how long you train. This is a classic silent bug: it often happens when a tensor is detached, converted to NumPy and back, or created outside the autograd graph. The network trains without error, but part of it is frozen.

After calling `loss.backward()`, every parameter that participated in the loss should have a `.grad` attribute that is not `None`. Inspecting these gradients, and their norms, is a direct check that learning can happen. A gradient norm that is a healthy nonzero number means the parameter is connected and receiving a learning signal; a norm of exactly zero or a value of `None` is a red flag.

## Overfit a Single Batch

The most valuable sanity check in all of deep learning is deliberately overfitting a single small batch. The logic is simple: a correctly built model and training loop, given just a handful of examples, should be able to memorize them completely and drive the loss to nearly zero. There is no generalization required, only memorization, and any network with enough capacity can memorize a few points.

If the loss on one batch will *not* go to near zero, then the problem is not your data, your learning rate schedule, or your regularization. The bug is in the model architecture or the training loop itself, and you have just isolated it to a tiny, fast experiment. If the loss *does* go to zero, you have confirmed that the forward pass, the loss function, the backward pass, and the optimizer step are all wired together correctly, which is exactly what you want to know before launching a long training run on the full dataset.

The cell below combines the gradient check and the single-batch overfit test. It runs the training loop on sixteen examples, prints the gradient norms after the first backward pass to confirm every parameter is connected, and watches the loss fall.

In [4]:
# The single-batch overfit test: a correct setup should drive the loss on
# one small batch to nearly zero. If it cannot, the bug is in the model or
# the training loop -- not in the data or the hyperparameters.
model = Net().to(device)
xb = X[:16].to(device)
yb = y[:16].to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(201):
    optimizer.zero_grad()
    loss = loss_fn(model(xb), yb)
    loss.backward()

    if epoch == 0:
        # After backward(), every parameter should have a gradient. A grad of
        # None means that parameter is disconnected from the loss.
        print("Gradient check after the first backward():")
        for name, p in model.named_parameters():
            status = "None (DISCONNECTED!)" if p.grad is None else f"norm = {p.grad.norm():.4f}"
            print(f"  {name:12s} {status}")
        print()

    optimizer.step()

    if epoch % 50 == 0:
        print(f"epoch {epoch:3d}   loss {loss.item():.6f}")

print("\nA loss near zero confirms the model and training loop can learn.")

Gradient check after the first backward():
  fc1.weight   norm = 0.0701
  fc1.bias     norm = 0.0263
  fc2.weight   norm = 0.1537
  fc2.bias     norm = 0.0675
  out.weight   norm = 0.2213
  out.bias     norm = 0.2349

epoch   0   loss 1.102862
epoch  50   loss 0.000151


epoch 100   loss 0.000056
epoch 150   loss 0.000044
epoch 200   loss 0.000036

A loss near zero confirms the model and training loop can learn.


The gradient check shows every parameter receiving a nonzero gradient, so nothing is disconnected. The loss then falls steadily toward zero as the network memorizes the sixteen examples. Seeing both of these confirms the training machinery is sound. Had the loss stalled at a high value, we would know to look at the model definition or the loop, not at the full dataset.

Together these techniques form a debugging discipline. Before trusting a long training run, print the model and confirm its output shape, use hooks to watch the intermediate activations if anything looks off, verify after a backward pass that every parameter has a gradient, and overfit a single batch to prove the whole pipeline can learn. Each check is fast and isolates a different class of bug. The next section, [Critical PyTorch Errors](t81_558_class_11_2_critical_errors.ipynb), turns from these silent failures to the loud ones: the specific exceptions PyTorch raises most often, how to read them, and how to fix them.